# Data Preprocessing

## Adult Census Income Dataset

### Objective

Data preprocessing is a crucial stage in the machine learning workflow. Raw datasets often contain missing values, duplicate records, inconsistent categorical values, and features that require transformation before model training.

This notebook applies preprocessing techniques to prepare the Adult Census Income dataset for machine learning.

The preprocessing steps include:

- Replacing placeholder values
- Handling missing values
- Removing duplicate records
- Cleaning categorical features
- Encoding categorical variables
- Scaling numerical features

The output of this notebook will be a clean and machine-learning-ready dataset.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

In [4]:
from src.data_loader import load_dataset

from src.preprocessing import (
    replace_placeholder_values,
    handle_missing_values,
    remove_duplicates,
    clean_categories,
    label_encode_column,
    ordinal_encode_columns,
    one_hot_encode,
    standard_scale,
    minmax_scale,
    get_numerical_columns,
    get_categorical_columns,
    dataset_summary,
)

In [5]:
path = "../datasets/adult/adult.data"

df = load_dataset(path)

df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## Missing Value Verification

The missing value handling process was successfully completed.

### Findings

- All missing values have been imputed.
- The dataset now contains **0 missing values**.
- The number of rows and columns remains unchanged because no records were removed during this step.
- The dataset is now ready for duplicate removal and further preprocessing.

The next preprocessing step is to identify and remove duplicate records.

In [8]:
df = handle_missing_values(df)
df.isnull().sum()


age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64

## Remove Duplicate Records

Duplicate records can introduce bias into a machine learning model by giving extra weight to repeated observations.

In this step, duplicate rows are identified and removed to improve the overall quality of the dataset.

Only exact duplicate rows are removed. No unique observations are affected.

In [12]:
print("Dataset Shape:", df.shape)

Dataset Shape: (32537, 15)


In [10]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate Rows Before Removal: {duplicate_count}")

df = remove_duplicates(df)

duplicate_count = df.duplicated().sum()

print(f"Duplicate Rows After Removal: {duplicate_count}")

Duplicate Rows Before Removal: 24
Duplicate Rows After Removal: 0


In [11]:
print("Dataset Shape:", df.shape)

Dataset Shape: (32537, 15)


## Duplicate Removal Findings

The dataset was examined for duplicate records, and all exact duplicate rows were removed successfully.

### Findings

- Duplicate rows before removal: **24**
- Duplicate rows after removal: **0**
- The dataset now contains **32,537 unique records**.
- The number of columns remains unchanged.

Removing duplicate records helps ensure that each observation contributes equally during model training and reduces the risk of biased learning.

## Clean Categorical Features

Categorical features may contain inconsistencies such as leading or trailing whitespace, mixed formatting, or other text-related issues.

Although the Adult Census Income dataset is generally well-structured, cleaning categorical features is a recommended preprocessing step to ensure consistency before encoding.

In this step, leading and trailing whitespace is removed from all categorical features.

No category names are changed, and no records are removed.

In [13]:
categorical_columns = get_categorical_columns(df)

print("Categorical Features:")
print(categorical_columns)

Categorical Features:
['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country', 'income']


In [14]:
df = clean_categories(df)

In [15]:
for column in categorical_columns:
    has_spaces = (
        df[column].astype(str)
        != df[column].astype(str).str.strip()
    ).any()

    print(f"{column}: {'Has Extra Spaces' if has_spaces else 'Clean'}")

workclass: Clean
education: Clean
marital_status: Clean
occupation: Clean
relationship: Clean
race: Clean
sex: Clean
native_country: Clean
income: Clean


In [16]:
dataset_summary(df)

{'Rows': 32537, 'Columns': 15, 'Missing Values': 0, 'Duplicate Rows': 0}

## Categorical Feature Cleaning Findings

All categorical features were inspected and standardized by removing any leading or trailing whitespace.

### Findings

- All categorical values are consistently formatted.
- No categories were renamed or removed.
- The number of rows and columns remains unchanged.
- The dataset is now ready for categorical feature encoding.

Cleaning categorical values helps prevent the creation of duplicate categories during encoding and ensures consistent feature representation.

# Encode Categorical Features

Machine learning algorithms require numerical input. Therefore, categorical features must be converted into numerical representations before model training.

Different encoding techniques are appropriate for different types of categorical variables.

This notebook demonstrates three commonly used encoding methods:

- Label Encoding
- Ordinal Encoding
- One-Hot Encoding

Each technique is demonstrated separately for educational purposes.

## Label Encoding

Label Encoding assigns a unique integer to each category.

It is commonly used for **binary categorical variables**.

Example:

Male → 1

Female → 0

In [17]:
label_df = df.copy()

label_df, sex_encoder = label_encode_column(
    label_df,
    "sex"
)

label_df[["sex"]].head()

,sex
0,1
1,1
2,1
3,1
4,0


## Ordinal Encoding

Ordinal Encoding is used for categorical variables that have a natural order.

Examples include education levels, satisfaction ratings, or skill levels.

Each category is mapped to an integer while preserving its relative ordering.

In [18]:
ordinal_df = df.copy()

ordinal_df, education_encoder = ordinal_encode_columns(
    ordinal_df,
    ["education"]
)

ordinal_df[["education"]].head()

,education
0,9.0
1,9.0
2,11.0
3,1.0
4,9.0


## One-Hot Encoding

One-Hot Encoding creates a separate binary feature for each category.

It is the preferred encoding technique for nominal categorical variables because it does not introduce artificial ordering.

In [19]:
onehot_columns = [
    "workclass",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "native_country"
]

onehot_df = one_hot_encode(
    df.copy(),
    onehot_columns
)

onehot_df.head()

,age,fnlwgt,education,education_num,sex,capital_gain,capital_loss,hours_per_week,income,workclass_Local-gov,...,native_country_Portugal,native_country_Puerto-Rico,native_country_Scotland,native_country_South,native_country_Taiwan,native_country_Thailand,native_country_Trinadad&Tobago,native_country_United-States,native_country_Vietnam,native_country_Yugoslavia
0,39,77516,Bachelors,13,Male,2174,0,40,<=50K,False,...,False,False,False,False,False,False,False,True,False,False
1,50,83311,Bachelors,13,Male,0,0,13,<=50K,False,...,False,False,False,False,False,False,False,True,False,False
2,38,215646,HS-grad,9,Male,0,0,40,<=50K,False,...,False,False,False,False,False,False,False,True,False,False
3,53,234721,11th,7,Male,0,0,40,<=50K,False,...,False,False,False,False,False,False,False,True,False,False
4,28,338409,Bachelors,13,Female,0,0,40,<=50K,False,...,False,False,False,False,False,False,False,False,False,False


In [20]:
print("Original Shape :", df.shape)
print("One-Hot Shape  :", onehot_df.shape)

Original Shape : (32537, 15)
One-Hot Shape  : (32537, 84)


## Encoding Results

The original dataset contained **15 features**, including both numerical and categorical variables.

After applying One-Hot Encoding to the selected categorical features, the dataset expanded to **84 features**.

### Observations

- Original dataset shape: **(32,537, 15)**
- Encoded dataset shape: **(32,537, 84)**
- The number of rows remained unchanged.
- The number of columns increased because each category was converted into its own binary feature.

### Why did the number of columns increase?

One-Hot Encoding creates a new binary column for each category within the selected categorical features. This transformation allows machine learning algorithms to process nominal categorical data without assuming any relationship or order between categories.

### Conclusion

The encoding process successfully transformed categorical variables into numerical representations while preserving all observations. The dataset is now suitable for machine learning algorithms that require numerical input.

# Scale Numerical Features

Many machine learning algorithms perform better when numerical features are on a similar scale.

Feature scaling reduces the effect of different measurement ranges and improves model convergence and performance.

This notebook demonstrates two widely used scaling techniques:

- Standard Scaling
- Min-Max Scaling

The scaling methods are demonstrated separately for comparison purposes.

In [21]:
numerical_columns = get_numerical_columns(df)

print(numerical_columns)

['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']


## Standard Scaling

Standard Scaling transforms numerical features so that they have:

- Mean ≈ 0
- Standard Deviation ≈ 1

This method is commonly used with algorithms such as:

- Logistic Regression
- Support Vector Machines (SVM)
- K-Nearest Neighbors (KNN)
- Neural Networks

In [22]:
standard_df = df.copy()

standard_df, standard_scaler = standard_scale(
    standard_df,
    numerical_columns
)

standard_df[numerical_columns].head()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
0,0.030390,-1.063569,1.134777,0.148292,-0.216743,-0.035664
1,0.836973,-1.008668,1.134777,-0.145975,-0.216743,-2.222483
2,-0.042936,0.245040,-0.420679,-0.145975,-0.216743,-0.035664
3,1.056950,0.425752,-1.198407,-0.145975,-0.216743,-0.035664
4,-0.776193,1.408066,1.134777,-0.145975,-0.216743,-0.035664


In [23]:
standard_df[numerical_columns].describe().round(2)

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,32537.00,32537.00,32537.00,32537.00,32537.00,32537.00
mean,-0.00,-0.00,0.00,0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.58,-1.68,-3.53,-0.15,-0.22,-3.19
25%,-0.78,-0.68,-0.42,-0.15,-0.22,-0.04
50%,-0.12,-0.11,-0.03,-0.15,-0.22,-0.04
75%,0.69,0.45,0.75,-0.15,-0.22,0.37
max,3.77,12.27,2.30,13.39,10.59,4.74


## Standard Scaling Results

Standard Scaling was successfully applied to all numerical features in the dataset.

### Observations

- The transformed numerical features have a **mean close to 0**.
- The **standard deviation is approximately 1** for every numerical feature.
- The number of rows and columns remains unchanged after scaling.
- Only the values of the numerical features have been transformed.

### Interpretation

Standard Scaling standardizes each numerical feature using the formula:

\[
z = \frac{x - \mu}{\sigma}
\]

where:

- \(x\) is the original value
- \(\mu\) is the mean of the feature
- \(\sigma\) is the standard deviation

As a result, all numerical features are placed on a comparable scale without changing their relative relationships.

### Conclusion

The StandardScaler successfully standardized the numerical features, making the dataset suitable for machine learning algorithms that are sensitive to feature scales, such as:

- Logistic Regression
- Support Vector Machines (SVM)
- K-Nearest Neighbors (KNN)
- Neural Networks

## Min-Max Scaling

Min-Max Scaling transforms numerical features into a fixed range between **0 and 1**.

This method preserves the relative relationships between observations and is commonly used in neural networks and distance-based algorithms.

In [24]:
minmax_df = df.copy()

minmax_df, minmax_scaler = minmax_scale(
    minmax_df,
    numerical_columns
)

minmax_df[numerical_columns].head()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
0,0.301370,0.044302,0.800000,0.02174,0.0,0.397959
1,0.452055,0.048238,0.800000,0.00000,0.0,0.122449
2,0.287671,0.138113,0.533333,0.00000,0.0,0.397959
3,0.493151,0.151068,0.400000,0.00000,0.0,0.397959
4,0.150685,0.221488,0.800000,0.00000,0.0,0.397959


In [25]:
minmax_df[numerical_columns].describe().round(2)

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,32537.00,32537.00,32537.00,32537.00,32537.00,32537.00
mean,0.30,0.12,0.61,0.01,0.02,0.40
std,0.19,0.07,0.17,0.07,0.09,0.13
min,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.15,0.07,0.53,0.00,0.00,0.40
50%,0.27,0.11,0.60,0.00,0.00,0.40
75%,0.42,0.15,0.73,0.00,0.00,0.45
max,1.00,1.00,1.00,1.00,1.00,1.00


## Min-Max Scaling Results

Min-Max Scaling was successfully applied to all numerical features.

### Observations

- Every numerical feature has been transformed to a range between **0 and 1**.
- The minimum value of each feature is **0**.
- The maximum value of each feature is **1**.
- The relative ordering of observations has been preserved.
- The number of rows and columns remains unchanged after scaling.

### Interpretation

Min-Max Scaling transforms each feature using the following equation:

\[
x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}
\]

where:

- \(x\) is the original value
- \(x_{min}\) is the minimum value of the feature
- \(x_{max}\) is the maximum value of the feature

This transformation rescales all numerical features into the interval **[0, 1]**, making them directly comparable while preserving their relative distances.

### Conclusion

Min-Max Scaling is particularly useful for machine learning algorithms that require normalized input data, such as:

- Neural Networks
- K-Nearest Neighbors (KNN)
- K-Means Clustering

The numerical features are now normalized and suitable for algorithms that are sensitive to feature magnitudes.

In [26]:
comparison = pd.DataFrame({
    "Original": df["age"].head().values,
    "StandardScaler": standard_df["age"].head().values,
    "MinMaxScaler": minmax_df["age"].head().values
})

comparison

,Original,StandardScaler,MinMaxScaler
0,39,0.030390,0.301370
1,50,0.836973,0.452055
2,38,-0.042936,0.287671
3,53,1.056950,0.493151
4,28,-0.776193,0.150685


## Comparison of Scaling Methods

The table above compares the original values of the **age** feature with the values obtained after applying Standard Scaling and Min-Max Scaling.

### Observations

| Original Value | StandardScaler | MinMaxScaler |
|---------------:|---------------:|-------------:|
| 39 | 0.030 | 0.301 |
| 50 | 0.837 | 0.452 |
| 38 | -0.043 | 0.288 |
| 53 | 1.057 | 0.493 |
| 28 | -0.776 | 0.151 |

### Interpretation

- **Original values** represent the actual ages in years.
- **StandardScaler** transforms the values so they are centered around a mean of 0 with a standard deviation of 1. Values greater than the mean become positive, while values below the mean become negative.
- **MinMaxScaler** rescales the values to the range **0 to 1**, where the smallest value becomes 0 and the largest value becomes 1.

### Conclusion

Both scaling techniques preserve the relative ordering of the observations but represent the data in different ways.

- Use **Standard Scaling** for algorithms that assume standardized input, such as Logistic Regression, Support Vector Machines (SVM), and Principal Component Analysis (PCA).
- Use **Min-Max Scaling** for algorithms that perform better with normalized inputs, such as Neural Networks, K-Nearest Neighbors (KNN), and K-Means Clustering.

The choice of scaling method depends on the requirements of the machine learning algorithm being used.

# Verify Final Dataset

## Objective

Before saving the processed dataset, a final verification is performed to ensure that all preprocessing steps have been applied successfully.

The verification checks:

- Dataset dimensions
- Missing values
- Duplicate records
- Data types
- Numerical and categorical features
- Overall dataset summary

This confirms that the dataset is clean and ready for feature engineering and machine learning model development.

In [27]:
print("Final Dataset Shape:", df.shape)

Final Dataset Shape: (32537, 15)


In [28]:
print("Total Missing Values:", df.isnull().sum().sum())

Total Missing Values: 0


In [29]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [30]:
dataset_summary(df)

{'Rows': 32537, 'Columns': 15, 'Missing Values': 0, 'Duplicate Rows': 0}

## Final Verification Results

The preprocessing workflow has been completed successfully.

### Summary

- All missing values have been handled.
- All duplicate records have been removed.
- Categorical features have been cleaned.
- Encoding techniques have been demonstrated.
- Numerical features have been scaled using both Standard Scaling and Min-Max Scaling.
- The dataset contains **32,537 observations** and **15 original features**, ready for the next stage of the machine learning workflow.

The dataset is now clean, consistent, and suitable for feature engineering and model development.

# Save Preprocessed Dataset

## Objective

After completing all preprocessing steps, the cleaned dataset is saved for future use.

Saving the processed dataset avoids repeating preprocessing in future notebooks and ensures that the same cleaned data is used throughout the machine learning workflow.

In [31]:
output_path = "../datasets/adult/adult_preprocessed.csv"

In [32]:
df.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print(f"Location: {output_path}")

Dataset saved successfully!
Location: ../datasets/adult/adult_preprocessed.csv


In [34]:
print("Saved Dataset Shape:", df.shape)

Saved Dataset Shape: (32537, 15)


## Dataset Saved Successfully

The cleaned dataset has been exported as a CSV file.

### File Information

- File Name: `adult_preprocessed.csv`
- Format: CSV
- Rows: 32,537
- Columns: 15

### Purpose

The saved dataset will serve as the cleaned version of the Adult Census Income dataset and can be reused in later stages of the project, such as feature engineering, model training, and evaluation.

Keeping a separate preprocessed dataset improves reproducibility and avoids repeating data cleaning steps.

# Save Preprocessing Pipeline

## Objective

Machine learning models must apply the same preprocessing steps to both training data and future unseen data.

Instead of recreating preprocessing objects every time, they are saved to disk and reused during model training and prediction.

This ensures consistency, reproducibility, and prevents data leakage.

In [35]:
import joblib
from pathlib import Path

In [36]:
model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)

In [37]:
joblib.dump(sex_encoder, model_dir / "label_encoder.pkl")

joblib.dump(education_encoder, model_dir / "ordinal_encoder.pkl")

joblib.dump(standard_scaler, model_dir / "standard_scaler.pkl")

joblib.dump(minmax_scaler, model_dir / "minmax_scaler.pkl")

['..\\models\\minmax_scaler.pkl']

In [38]:
for file in model_dir.iterdir():
    print(file.name)

label_encoder.pkl
minmax_scaler.pkl
ordinal_encoder.pkl
standard_scaler.pkl



## Preprocessing Pipeline Saved

The preprocessing objects have been successfully saved for future use.

### Saved Components

- Label Encoder
- Ordinal Encoder
- Standard Scaler
- Min-Max Scaler

### Why Save Them?

Using the same preprocessing objects during training and inference ensures that data is transformed consistently.

This prevents discrepancies between training and prediction and supports reproducible machine learning workflows.

# Day 4 Summary

The Adult Census Income dataset has been successfully prepared for machine learning.

## Completed Tasks

- Loaded the dataset
- Explored the data
- Performed statistical analysis
- Conducted data quality assessment
- Handled missing values
- Removed duplicate records
- Cleaned categorical features
- Demonstrated Label Encoding
- Demonstrated Ordinal Encoding
- Demonstrated One-Hot Encoding
- Applied Standard Scaling
- Applied Min-Max Scaling
- Verified the final dataset
- Saved the cleaned dataset
- Saved preprocessing objects for future use

## Outcome

The dataset is now clean, consistent, and ready for feature engineering, model training, and evaluation.

The saved preprocessing objects ensure that the same transformations can be applied consistently to future datasets, following professional machine learning engineering practices.